# Building a Dashboard

Combine several FRED series — the policy rate, year-over-year inflation, and the unemployment rate — into one `TimeArray` and plot them. Using **TimeSeries.jl** keeps the series time-indexed and needs no DataFrames dependency.

In [ ]:
Pollis.packages("FredData", "TimeSeries", "Plots")

In [ ]:
# ENV["FRED_API_KEY"] = "YOURKEY"  # uncomment and set if not already in the environment
using FredData, TimeSeries

f = Fred()

# Fetch a FRED series straight into a TimeArray (s.data has :date and :value)
function series(id, name; kwargs...)
    s = get_data(f, id; kwargs...)
    return TimeArray(s.data.date, s.data.value, [name])
end

## Download the Series

Pull each series from a common start date. `units="pc1"` turns CPI into year-over-year inflation.

In [ ]:
start = "2000-01-01"

funds  = series("FEDFUNDS", :FedFunds; observation_start = start)
cpi    = series("CPIAUCSL", :CPI_YoY;  units = "pc1", observation_start = start)
unrate = series("UNRATE",   :Unemploy; observation_start = start)

length(funds), length(cpi), length(unrate)

## Merge into One Panel

All three series are monthly, so `merge` aligns them on date (an inner join on the timestamp) into a single three-column `TimeArray`.

In [ ]:
panel = merge(funds, cpi, unrate)
last(panel, 6)

## Plot

TimeSeries provides Plots recipes, so `plot(panel)` charts every column at once.

In [ ]:
using Plots

plot(panel; label = ["Fed funds rate" "CPI inflation (YoY)" "Unemployment rate"])
xlabel!("Date"); ylabel!("Percent")

## Summary
- `series(id, name)` returns each FRED series as a `TimeArray` — no DataFrames needed.
- All three series are monthly, so `merge` aligns them into one panel on the shared dates.
- TimeSeries has Plots recipes, so `plot(panel)` charts every column at once.